# Clase 8 — Atención y Transformers

En esta clase construyes, paso a paso y en directo, el mecanismo que sostiene a los modelos de lenguaje actuales: la atención (*attention*). Retomas el problema que dejaste abierto en la Clase 6 con las redes neuronales recurrentes (RNN, *recurrent neural network*) y llegas hasta la pieza central del Transformer.

**Objetivos de aprendizaje**

- Explicar por qué el cuello de botella de las RNN limita el procesamiento de secuencias largas
- Relacionar los conceptos de consulta, clave y valor con una analogía cotidiana
- Implementar en PyTorch la atención escalada por producto punto usando solo `matmul` y `softmax`
- Visualizar e interpretar un mapa de calor de pesos de atención
- Situar la atención multi-cabeza como pieza central del Transformer

## El cuello de botella de las RNN

Recuerda la Clase 6: una red neuronal recurrente (RNN) o su variante más robusta, la LSTM (memoria a corto y largo plazo, *long short-term memory*), procesa una secuencia paso a paso y actualiza un estado oculto de tamaño fijo. Ese estado tiene que resumir toda la frase, tenga cinco palabras o cincuenta. Cuanto más larga es la secuencia, más información se pierde por el camino, casi siempre la que apareció al principio.

La atención resuelve el problema de otra forma: en lugar de comprimir todo en un único vector, deja que el modelo vuelva a consultar la secuencia completa en cada paso y decida qué partes importan ahora mismo.

## Consulta, clave y valor

Imagina que buscas un libro en una biblioteca enorme. Llevas una pregunta concreta en la cabeza: es la consulta (*query*). Recorres los lomos de los libros, cada uno con una etiqueta que resume su contenido: son las claves (*keys*). Comparas tu consulta con cada clave y decides cuánto se parecen. No te llevas la etiqueta a casa: te llevas el contenido del libro más relevante, es decir, su valor (*value*).

Un mecanismo de atención hace exactamente esto con vectores: compara una consulta con un conjunto de claves, obtiene un peso de relevancia para cada una mediante la función softmax, y combina los valores ponderados por esos pesos.

## Atención escalada por producto punto desde cero

Implementas ahora la fórmula exacta:

$$\text{Atención}(Q, K, V) = \text{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}\right) V$$

Generas $Q$, $K$ y $V$ como tres proyecciones lineales distintas de una misma entrada $X$. Calculas las puntuaciones con `matmul` (el operador `@`), las escalas por la raíz de $d_k$ y aplicas softmax para obtener los pesos de atención, que después ponderan los valores. Trabajas sobre una frase de juguete, ya tokenizada a mano.

In [ ]:
import torch

torch.manual_seed(0)

# Frase de ejemplo, ya tokenizada en palabras (en un modelo real usarías un tokenizador real)
tokens = ["el", "gato", "negro", "duerme", "en", "el", "sofa"]
n = len(tokens)
d = 8      # dimensión del embedding, pequeña solo para que el ejemplo sea legible
d_k = 8    # dimensión de consultas y claves

# Embeddings de partida: en un modelo entrenado vendrían de una capa de embedding aprendida;
# aquí los generamos al azar para centrarnos en el mecanismo de atención
X = torch.randn(n, d)

# Proyecciones lineales aprendibles: transforman el mismo embedding en tres papeles distintos
W_q = torch.randn(d, d_k) * 0.5
W_k = torch.randn(d, d_k) * 0.5
W_v = torch.randn(d, d_k) * 0.5

Q = X @ W_q   # consultas: qué busca cada palabra
K = X @ W_k   # claves: qué ofrece cada palabra al ser comparada
V = X @ W_v   # valores: qué contenido entrega cada palabra si es elegida


def softmax_manual(z, dim=-1):
    # Restas el máximo antes de la exponencial por estabilidad numérica,
    # el mismo truco que viste con softmax en la capa de salida de clasificación
    z = z - z.max(dim=dim, keepdim=True).values
    exp_z = torch.exp(z)
    return exp_z / exp_z.sum(dim=dim, keepdim=True)


def atencion(Q, K, V):
    d_k = Q.shape[-1]
    puntuaciones = Q @ K.transpose(-2, -1)          # producto punto: consulta contra cada clave
    puntuaciones = puntuaciones / (d_k ** 0.5)       # escalado por raíz de d_k
    pesos = softmax_manual(puntuaciones, dim=-1)     # cada fila suma uno: es una distribución
    salida = pesos @ V                               # combinación ponderada de los valores
    return salida, pesos


salida, pesos_atencion = atencion(Q, K, V)

print("Forma de la salida:", salida.shape)
print("Cada fila de pesos suma:", pesos_atencion.sum(dim=-1))
print("Pesos de atención:\n", pesos_atencion)

## Visualiza dónde mira el modelo

Los pesos de atención forman una matriz de tamaño n×n: la palabra que pregunta (consulta) en un eje, la palabra consultada (clave) en el otro. Dibujas esa matriz como un mapa de calor con `imshow`: los tonos más claros marcan los pesos más altos. Como los embeddings de este ejemplo son aleatorios, no esperes un patrón lingüístico real, sino la mecánica pura del mecanismo.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.imshow(pesos_atencion.detach().numpy(), cmap="viridis")
plt.xticks(range(n), tokens, rotation=45, ha="right")
plt.yticks(range(n), tokens)
plt.xlabel("Clave (palabra observada)")
plt.ylabel("Consulta (palabra que pregunta)")
plt.title("Pesos de atención escalada por producto punto")
plt.colorbar(label="Peso de atención")
plt.tight_layout()
plt.show()

## De la atención al Transformer

Una sola atención mezcla en un único patrón de pesos todo tipo de relaciones: sintácticas, semánticas, de proximidad. El Transformer ejecuta varias atenciones en paralelo, cada una con sus propias proyecciones $W_i^Q$, $W_i^K$, $W_i^V$, entrenadas de forma independiente: es la atención multi-cabeza (*multi-head attention*). Concatena las salidas de las h cabezas y las proyecta de nuevo a la dimensión original con una matriz $W^O$ aprendida.

$$\text{cabeza}_i = \text{Atención}(QW_i^Q, KW_i^K, VW_i^V)$$
$$\text{MultiCabeza}(Q,K,V) = \text{Concat}(\text{cabeza}_1,\dots,\text{cabeza}_h)\,W^O$$

Cada cabeza tiende a especializarse: una atiende a palabras vecinas, otra captura concordancia gramatical a larga distancia, otra sigue referencias pronominales. El Transformer apila varios bloques de este tipo, uno tras otro, y prescinde por completo de la recurrencia y de las convoluciones. Hoy no lo implementas entero: en la próxima clase construyes un bloque de Transformer completo, con atención multi-cabeza, normalización y una red hacia adelante, apoyándote directamente en el código que acabas de escribir.

## Resumen de puntos clave

- Las RNN y las LSTM sufren un cuello de botella: comprimen una secuencia entera en un único estado oculto de tamaño fijo, y pierden información en secuencias largas
- La atención sustituye ese resumen único por un vector de contexto calculado en cada paso, como suma ponderada de toda la entrada
- La consulta, la clave y el valor son tres proyecciones lineales distintas de la misma entrada: la clave decide dónde mirar, el valor es lo que efectivamente lees
- La atención escalada por producto punto se calcula como softmax(QKᵀ/√d_k)V; el escalado evita que el softmax sature con dimensiones grandes
- La atención multi-cabeza ejecuta varias atenciones en paralelo y deja que cada cabeza se especialice en un tipo distinto de relación entre palabras
- El Transformer apila bloques de autoatención (*self-attention*) y prescinde por completo de la recurrencia

**Abre ahora el notebook de práctica companion, `../notebooks_practica/clase08_practica_atencion_transformers.ipynb`, y pon en práctica lo que acabas de ver.**